##BRONZE STREAMING TABLE

In [0]:

import dlt

from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
  name="bronze_hotel")
def bronze_hotel():
  return spark.readStream.table("ingestion_catalog.streaming_dlt.hotel")

##SILVER VIEW

In [0]:

@dlt.view
def silver_hotel_view():
    df = dlt.read("bronze_hotel")
    df_new = df.withColumn("package",lit("southern_travels")).withColumn("Number of days stayed",datediff(df.checkout_date,df.checkin_date))
    return df_new


###GOLD MATVIEW

In [0]:
@dlt.table
def gold_hotel_aggmatview():
    df = dlt.read("silver_hotel_view")
    df_new = df.groupBy("room_tye").agg(count("name")).select("room_tye","count(name)").withColumnRenamed("count(name)","Number_of_bookings")
    return df_new